# Sentiment Analysis on IMDB Movie Reviews

**Objective:** Build a sentiment analysis model that classifies movie reviews as positive or negative using Natural Language Processing (NLP) techniques.

## 1. Import Libraries

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Text preprocessing
import re
import nltk
from nltk.corpus import stopwords

# Feature extraction and modeling
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Download NLTK stopwords
nltk.download('stopwords', quiet=True)

print('All libraries imported successfully!')

All libraries imported successfully!


## 2. Load and Inspect the Dataset

In [3]:
# Load the IMDB dataset
df = pd.read_csv('..\docs\IMDB Dataset.csv')

# Display basic info
print(f'Dataset shape: {df.shape}')
print(f'\nFirst 5 rows:')
display(df.head())

print(f'\nColumn names: {df.columns.tolist()}')
print(f'\nData types:')
print(df.dtypes)

Dataset shape: (999, 2)

First 5 rows:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive



Column names: ['review', 'sentiment']

Data types:
review       object
sentiment    object
dtype: object


In [4]:
# Check class balance
print('Sentiment distribution:')
print(df['sentiment'].value_counts())
print(f'\nPercentage:')
print(df['sentiment'].value_counts(normalize=True) * 100)

Sentiment distribution:
sentiment
positive    501
negative    498
Name: count, dtype: int64

Percentage:
sentiment
positive    50.15015
negative    49.84985
Name: proportion, dtype: float64


In [5]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())

# Check for duplicates
print(f'\nDuplicate reviews: {df.duplicated().sum()}')

Missing values:
review       0
sentiment    0
dtype: int64

Duplicate reviews: 0


In [6]:
# Remove duplicates if any
df = df.drop_duplicates()
print(f'Shape after removing duplicates: {df.shape}')

Shape after removing duplicates: (999, 2)


## 3. Text Preprocessing

In [7]:
# Define text cleaning function
def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove HTML tags (like <br />)
    text = re.sub(r'<[^>]+>', '', text)
    # Remove punctuation and special characters (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning to the review column
df['cleaned_review'] = df['review'].apply(clean_text)

# Show before and after
print('Original review:')
print(df['review'].iloc[0][:200])
print('\nCleaned review:')
print(df['cleaned_review'].iloc[0][:200])

Original review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo

Cleaned review:
one of the other reviewers has mentioned that after watching just oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brut


In [8]:
# Encode sentiment labels: positive -> 1, negative -> 0
df['sentiment_label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print('Label mapping:')
print(df[['sentiment', 'sentiment_label']].head(10))

Label mapping:
  sentiment  sentiment_label
0  positive                1
1  positive                1
2  positive                1
3  negative                0
4  positive                1
5  positive                1
6  positive                1
7  negative                0
8  negative                0
9  positive                1


## 4. Convert Text to Numerical Features using TF-IDF

In [9]:
# Initialize TF-IDF Vectorizer
# max_features=5000 limits to the 5000 most important words
# ngram_range=(1,2) includes both unigrams and bigrams
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english'
)

# Fit and transform the cleaned reviews
X = vectorizer.fit_transform(df['cleaned_review'])
y = df['sentiment_label'].values

print(f'Feature matrix shape: {X.shape}')
print(f'Number of samples: {X.shape[0]}')
print(f'Number of features (words/bigrams): {X.shape[1]}')

Feature matrix shape: (999, 5000)
Number of samples: 999
Number of features (words/bigrams): 5000


## 5. Split Data into Training and Testing Sets

In [10]:
# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size: {X_train.shape[0]}')
print(f'Testing set size: {X_test.shape[0]}')
print(f'\nTraining set class distribution:')
print(f'  Positive: {sum(y_train == 1)}')
print(f'  Negative: {sum(y_train == 0)}')
print(f'\nTesting set class distribution:')
print(f'  Positive: {sum(y_test == 1)}')
print(f'  Negative: {sum(y_test == 0)}')

Training set size: 799
Testing set size: 200

Training set class distribution:
  Positive: 401
  Negative: 398

Testing set class distribution:
  Positive: 100
  Negative: 100


## 6. Train Classification Models

In [11]:
# Model 1: Logistic Regression
print('Training Logistic Regression...')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
print('Logistic Regression trained successfully!')

Training Logistic Regression...
Logistic Regression trained successfully!


In [12]:
# Model 2: Multinomial Naive Bayes
print('Training Multinomial Naive Bayes...')
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
print('Naive Bayes trained successfully!')

Training Multinomial Naive Bayes...
Naive Bayes trained successfully!


## 7. Evaluate Models

In [13]:
# Predictions
y_pred_lr = lr_model.predict(X_test)
y_pred_nb = nb_model.predict(X_test)

# Evaluation metrics
print('=' * 60)
print('MODEL EVALUATION RESULTS')
print('=' * 60)

# Logistic Regression
print('\n--- Logistic Regression ---')
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
print(f'Accuracy:  {lr_accuracy:.4f}')
print(f'F1-Score:  {lr_f1:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_lr))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

# Naive Bayes
print('\n--- Multinomial Naive Bayes ---')
nb_accuracy = accuracy_score(y_test, y_pred_nb)
nb_f1 = f1_score(y_test, y_pred_nb)
print(f'Accuracy:  {nb_accuracy:.4f}')
print(f'F1-Score:  {nb_f1:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_nb))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_nb, target_names=['Negative', 'Positive']))

MODEL EVALUATION RESULTS

--- Logistic Regression ---
Accuracy:  0.7950
F1-Score:  0.8038

Confusion Matrix:
[[75 25]
 [16 84]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.82      0.75      0.79       100
    Positive       0.77      0.84      0.80       100

    accuracy                           0.80       200
   macro avg       0.80      0.79      0.79       200
weighted avg       0.80      0.80      0.79       200


--- Multinomial Naive Bayes ---
Accuracy:  0.7750
F1-Score:  0.7644

Confusion Matrix:
[[82 18]
 [27 73]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.75      0.82      0.78       100
    Positive       0.80      0.73      0.76       100

    accuracy                           0.78       200
   macro avg       0.78      0.77      0.77       200
weighted avg       0.78      0.78      0.77       200



In [14]:
# Compare models
print('=' * 60)
print('MODEL COMPARISON')
print('=' * 60)
print(f'{"Model":<30} {"Accuracy":<15} {"F1-Score":<15}')
print('-' * 60)
print(f'{"Logistic Regression":<30} {lr_accuracy:<15.4f} {lr_f1:<15.4f}')
print(f'{"Multinomial Naive Bayes":<30} {nb_accuracy:<15.4f} {nb_f1:<15.4f}')

# Select the best model
if lr_f1 >= nb_f1:
    best_model = lr_model
    best_model_name = 'Logistic Regression'
else:
    best_model = nb_model
    best_model_name = 'Multinomial Naive Bayes'

print(f'\nBest model: {best_model_name}')

MODEL COMPARISON
Model                          Accuracy        F1-Score       
------------------------------------------------------------
Logistic Regression            0.7950          0.8038         
Multinomial Naive Bayes        0.7750          0.7644         

Best model: Logistic Regression


## 8. Test on Custom Example Sentences

In [15]:
# Define custom test sentences
custom_sentences = [
    "This was the best experience ever! I absolutely loved every moment of it.",
    "This was a complete waste of time. Terrible acting and boring plot.",
    "The movie was okay, not great but not terrible either. Some parts were good."
]

# Clean the custom sentences
cleaned_custom = [clean_text(sentence) for sentence in custom_sentences]

# Transform using the same vectorizer
custom_features = vectorizer.transform(cleaned_custom)

# Predict using the best model
custom_predictions = best_model.predict(custom_features)
custom_probabilities = best_model.predict_proba(custom_features)

# Display results
print('=' * 80)
print('CUSTOM SENTENCE PREDICTIONS')
print('=' * 80)

for i, sentence in enumerate(custom_sentences):
    sentiment = 'POSITIVE' if custom_predictions[i] == 1 else 'NEGATIVE'
    prob_positive = custom_probabilities[i][1]
    prob_negative = custom_probabilities[i][0]
    print(f'\nSentence {i+1}: "{sentence}"')
    print(f'Predicted Sentiment: {sentiment}')
    print(f'Confidence: Positive={prob_positive:.4f}, Negative={prob_negative:.4f}')

CUSTOM SENTENCE PREDICTIONS

Sentence 1: "This was the best experience ever! I absolutely loved every moment of it."
Predicted Sentiment: POSITIVE
Confidence: Positive=0.7437, Negative=0.2563

Sentence 2: "This was a complete waste of time. Terrible acting and boring plot."
Predicted Sentiment: NEGATIVE
Confidence: Positive=0.2170, Negative=0.7830

Sentence 3: "The movie was okay, not great but not terrible either. Some parts were good."
Predicted Sentiment: POSITIVE
Confidence: Positive=0.5648, Negative=0.4352


## 9. Final Summary

In [16]:
print('=' * 80)
print('FINAL SUMMARY')
print('=' * 80)

print('''
PIPELINE OVERVIEW
-----------------
1. Data Loading: Loaded the IMDB movie reviews dataset (999 reviews).
2. Text Preprocessing: 
   - Converted text to lowercase
   - Removed HTML tags (<br />)
   - Removed punctuation and special characters
   - Removed extra whitespace
3. Feature Extraction: Used TF-IDF Vectorization with:
   - max_features=5000 (top 5000 words/bigrams)
   - ngram_range=(1,2) (unigrams and bigrams)
   - English stop words removed
4. Train/Test Split: 80% training, 20% testing (stratified)
5. Model Training: Trained two classifiers:
   - Logistic Regression
   - Multinomial Naive Bayes
6. Evaluation: Used Accuracy and F1-Score metrics
''')

print(f'MODEL PERFORMANCE')
print(f'-----------------')
print(f'Best Model: {best_model_name}')
print(f'Test Accuracy: {max(lr_accuracy, nb_accuracy):.4f} ({max(lr_accuracy, nb_accuracy)*100:.2f}%)')
print(f'Test F1-Score: {max(lr_f1, nb_f1):.4f}')

print('''
LIMITATIONS
-----------
1. Sarcasm Detection: The model struggles with sarcastic or ironic statements
   where the literal meaning differs from the intended sentiment.
2. Context Understanding: TF-IDF treats words independently and cannot
   capture complex contextual relationships or negation well (e.g., "not bad"
   might still be classified as negative due to the word "bad").
3. Vocabulary Limitation: With max_features=5000, rare but important words
   may be excluded from the feature set.
4. Domain Specificity: The model is trained on movie reviews and may not
   generalize well to other domains like product reviews or tweets.
5. No Word Order: Bag-of-words approach loses word order information,
   which can be important for understanding sentiment.
''')

FINAL SUMMARY

PIPELINE OVERVIEW
-----------------
1. Data Loading: Loaded the IMDB movie reviews dataset (999 reviews).
2. Text Preprocessing: 
   - Converted text to lowercase
   - Removed HTML tags (<br />)
   - Removed punctuation and special characters
   - Removed extra whitespace
3. Feature Extraction: Used TF-IDF Vectorization with:
   - max_features=5000 (top 5000 words/bigrams)
   - ngram_range=(1,2) (unigrams and bigrams)
   - English stop words removed
4. Train/Test Split: 80% training, 20% testing (stratified)
5. Model Training: Trained two classifiers:
   - Logistic Regression
   - Multinomial Naive Bayes
6. Evaluation: Used Accuracy and F1-Score metrics

MODEL PERFORMANCE
-----------------
Best Model: Logistic Regression
Test Accuracy: 0.7950 (79.50%)
Test F1-Score: 0.8038

LIMITATIONS
-----------
1. Sarcasm Detection: The model struggles with sarcastic or ironic statements
   where the literal meaning differs from the intended sentiment.
2. Context Understanding: TF-IDF